# Deploy a scikit-learn model on TrueFoundry with MLServer (Seldon), no inference code

- Flow: **train → log model → package → build & push image → deploy → infer**
- No inference code is written by hand — the runtime is generated or supplied
- Here we own the image: build locally, push to Docker Hub, deploy that tag. TrueFoundry never builds.
- Docs: https://docs.seldon.ai/mlserver

---

## 0. Setup

> **Kernel: `Python 3.12 (TFY MLServer)`** — venv at `~/work/.venvs/tfy-mlserver-py312`.
> Also not a preference: `mlserver` is `requires_python <3.13`, so `mlserver build` cannot run
> on 3.14. Kept separate from the Triton env, which pins numpy < 2.

In [ ]:
# %pip install -U truefoundry scikit-learn mlserver mlserver-sklearn joblib requests

In [ ]:
import json
import os
import time
from pathlib import Path

import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

from truefoundry.ml import get_client
from truefoundry.common.session import Session
from truefoundry_sdk import TrueFoundry

In [ ]:
TFY_HOST = "https://internal.devtest.truefoundry.tech"
ML_REPO = "default"
CLUSTER = "tfy-usea1-devtest"
WORKSPACE = "seldon-tf"
DOMAIN = "tfy-usea1-ctl.devtest.truefoundry.tech"
WS_FQN = f"{CLUSTER}:{WORKSPACE}"

DOCKER_USER = "mkozubal"
TAG = "1"
PORT = 8080
READY_TIMEOUT = 300

NAME = "iris-sk-mlserver"
SVC = f"{NAME}-svc"
IMAGE = f"{DOCKER_USER}/{SVC}:{TAG}"
HOST = f"{SVC}-{WORKSPACE}-{PORT}.{DOMAIN}"

## 1. Train and save

In [ ]:
Path(SVC).mkdir(exist_ok=True)

X, y = load_iris(return_X_y=True)
model = RandomForestClassifier(n_estimators=10).fit(X, y)
joblib.dump(model, Path(SVC, "model.joblib"))

## 2. Describe the model for MLServer

No schema and no generated server. `model-settings.json` names the runtime, and MLServer's
`SKLearnModel` loads the joblib itself. The class is `SKLearnModel` — not `SKLearnRuntime`.

In [ ]:
Path(SVC, "model-settings.json").write_text(json.dumps({
    "name": SVC,
    "implementation": "mlserver_sklearn.SKLearnModel",
    "parameters": {"uri": "./model.joblib"},
}, indent=2))

Path(SVC, "requirements.txt").write_text("mlserver-sklearn==1.7.1\n")

## 3. Log to the registry — no schema

Versioning and lineage only: no framework, so no schema and no `deploy-init`.
`mlserver build` is what turns the folder into a server.

In [ ]:
client = get_client()

FQN = client.log_model(ml_repo=ML_REPO, name=NAME, model_file_or_folder=SVC).fqn
print(FQN)

## 4. Build and push the image

`mlserver build` writes its own Dockerfile (base `seldonio/mlserver:*-slim`) and its `COPY . .`
**bakes the model into the image** — the Seldon way, so no `artifacts_download` when we deploy.

It has no `--platform` flag, but it shells out to `docker build`, so `DOCKER_DEFAULT_PLATFORM`
gets us the amd64 image the cluster needs.

In [ ]:
os.environ["DOCKER_DEFAULT_PLATFORM"] = "linux/amd64"

!cd {SVC} && mlserver build . -t {IMAGE}
!docker push {IMAGE}

## 5. Deploy as a service

Prebuilt image, so `"image": {"type": "image", ...}` instead of a build.
The model is already inside the image, so there is no `artifacts_download`.

Auth reuses the `tfy login` session, so no PAT in the notebook.

`create_or_update` returns a **deployment**, so delete later with `app.data.application_id`,
not `app.data.id`.

In [ ]:
tfy = TrueFoundry(base_url=TFY_HOST, api_key=Session.new().access_token)

app = tfy.applications.create_or_update(manifest={
    "name": SVC, "type": "service", "workspace_fqn": WS_FQN,
    "image": {"type": "image", "image_uri": IMAGE},
    "ports": [{"port": PORT, "protocol": "TCP", "expose": True,
               "app_protocol": "http", "host": HOST}],
    "resources": {"cpu_request": 0.5, "cpu_limit": 0.5,
                  "memory_request": 1000, "memory_limit": 1000,
                  "ephemeral_storage_request": 2000, "ephemeral_storage_limit": 2000},
    "replicas": 1,
})
print(SVC, app.data.application_id)

## 6. Call the model

MLServer speaks the KServe v2 protocol on port 8080: `POST /v2/models/<name>/infer`.
`data` is **flat** row-major alongside `shape` (Triton also accepts nested), and the input
name is the runtime's method, `predict`.

In [ ]:
deadline = time.time() + READY_TIMEOUT
while time.time() < deadline:
    try:
        if requests.get(f"https://{HOST}/v2/models/{SVC}/ready", timeout=5).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError(f"{SVC} not ready after {READY_TIMEOUT}s")

print(SVC, "ready")

In [ ]:
r = requests.post(
    f"https://{HOST}/v2/models/{SVC}/infer",
    json={"inputs": [{"name": "predict", "shape": [1, 4], "datatype": "FP32",
                      "data": [5.1, 3.5, 1.4, 0.2]}]},
    timeout=30,
)
print(r.status_code, r.text)

The same call as a copy-pasteable `curl`, for testing outside the notebook.

In [ ]:
print(f'''curl -X POST https://{HOST}/v2/models/{SVC}/infer \\
  -H 'Content-Type: application/json' \\
  -d '{{"inputs": [{{"name": "predict", "shape": [1, 4], "datatype": "FP32", "data": [5.1, 3.5, 1.4, 0.2]}}]}}' ''')

## Cleanup

Commented out so a re-run does not wipe a service you still want. Uncomment to remove the
service, the model version, the local folder and the local image.

In [ ]:
# tfy.applications.delete(id=app.data.application_id)
# client.get_model_version_by_fqn(FQN).delete()

# !rm -rf {SVC}
# !docker rmi -f {IMAGE}